kiiling app2: 
pkill -f "/home/manojcloudvm/infra-lab/app2/app.py

manojcloudvm@instance-20260404-024439:~/infra-lab/app2$ curl -i "http://127.0.0.1/proxy-dep"
HTTP/1.1 502 BAD GATEWAY
Server: nginx/1.22.1
Date: Tue, 07 Apr 2026 00:30:02 GMT
Content-Type: application/json
Content-Length: 315
Connection: keep-alive

{
  "error": --------->>"HTTPConnectionPool(host='127.0.0.1', port=5001)<------->>: Max retries exceeded with url: /upstream-ok (Caused by NewConnectionError(\"HTTPConnection(host='127.0.0.1', port=5001): Failed to establish a new connection: [Errno 111] Connection refused\"))",<----------
  "upstream_url": "http://127.0.0.1:5001/upstream-ok"
}

ss -tulpn | grep 5001 ----> shows nothing

or 

lsof -i :5001

Checking working app's logs


> tail -n 50 ~/infra-lab/app1/app1.log-<-------- (make sure they are latest)

127.0.0.1 - - [07/Apr/2026 00:01:54] "GET /containers/json HTTP/1.1" 404 -
127.0.0.1 - - [07/Apr/2026 00:04:40] "GET /proxy-dep HTTP/1.1" 502 -
127.0.0.1 - - [07/Apr/2026 00:05:12] "GET /proxy-dep HTTP/1.1" 502 -
127.0.0.1 - - [07/Apr/2026 00:11:53] "GET /proxy-dep HTTP/1.1" 502 -
127.0.0.1 - - [07/Apr/2026 00:21:05] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 00:26:23] "POST /GponForm/diag_Form?images/ HTTP/1.1" 404 -
127.0.0.1 - - [07/Apr/2026 00:29:09] "GET /.env HTTP/1.1" 404 -
127.0.0.1 - - [07/Apr/2026 00:29:09] "POST / HTTP/1.1" 405 -
127.0.0.1 - - [07/Apr/2026 00:30:02] "GET /proxy-dep HTTP/1.1" 502 -
127.0.0.1 - - [07/Apr/2026 00:34:59] "GET /proxy-dep HTTP/1.1" 502 -


keep hitting: curl -i "http://127.0.0.1/proxy-dep"

you keep getting: "GET /proxy-dep HTTP/1.1" 502s 

nohup ~/infra-lab/app2/venv/bin/python ~/infra-lab/app2/app.py > ~/infra-lab/app2/app2.log 2>&1 &


if you do not know, where app arch, you would involve the app owner to know what does /proxy-dep call and what supposed to be on ip:5001

502 means

Bad Gateway usually means:

a proxy, gateway, or intermediary got an invalid/bad response from an upstream server

Common cases:

upstream service is down
upstream port not listening
upstream timed out or reset connection
proxy is pointing to wrong host/port
TLS/problem between proxy and upstream
malformed upstream response


Interview Words: A 502 usually tells me the component handling the request is up, but it had trouble getting a valid response from an upstream dependency. I would verify whether the upstream is running, reachable on the expected host/port, and responding correctly.

